# Module `models.py`

Ce notebook presente les structures de donnees centrales du projet CESIPATH. Ces objets sont le vocabulaire commun entre le generateur, les solveurs, la simulation dynamique et la visualisation.

Tout est en dataclasses immuables ou legeres pour eviter les effets de bord.

## Pourquoi commencer par les modeles ?

Separer les structures du comportement permet :

- aux solveurs de ne dependre que d'un `SolverInput` (construit a partir de ces modeles) ;
- au generateur de muter des dict sans casser des objets immuables ;
- au validateur de raisonner sur une instance figee.

Les cinq objets principaux :

| Objet | Role |
|---|---|
| `EdgeStatus` | Enum a 3 valeurs : libre / surcout / interdit |
| `EdgeAttributes` | Cout de base, statut et surcout statique d'une arete |
| `GraphGenerationConfig` | Tous les parametres modifiables |
| `GraphInstance` | Instance complete prete a exploiter |
| `DynamicGraphSnapshot` | Etat du reseau a un tour donne |

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.models import (
    DynamicGraphSnapshot,
    EdgeAttributes,
    EdgeStatus,
    GraphGenerationConfig,
    GraphInstance,
)

## 1. `EdgeStatus`

Enum a trois valeurs qui decrit l'etat **statique** d'une arete :

- `FREE` (`libre`) : arete utilisable, sans surcout ;
- `SURCHARGED` (`surcout`) : arete utilisable mais plus couteuse ;
- `FORBIDDEN` (`interdit`) : arete retiree du graphe residuel.

L'enum est un `str` enum, donc `EdgeStatus.FREE.value == "libre"`. Cela simplifie la serialisation.

In [2]:
for status in EdgeStatus:
    print(status.name, "->", status.value)

FREE -> libre
SURCHARGED -> surcout
FORBIDDEN -> interdit


## 2. `EdgeAttributes`

Dataclass frozen. Trois champs :

- `base_cost` : cout physique minimal (distance euclidienne) ;
- `status` : valeur d'`EdgeStatus` ;
- `static_surcharge` : coefficient de surcout applique si `status=SURCHARGED`.

La propriete `static_cost` est calculee :

```text
static_cost = base_cost * (1 + static_surcharge)   si FREE ou SURCHARGED
static_cost = +inf                                 si FORBIDDEN
```

Consequence : une arete FORBIDDEN a un cout infini, ce qui l'elimine naturellement de Dijkstra.

In [3]:
free_edge = EdgeAttributes(base_cost=10.0)
surcharged_edge = EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25)
forbidden_edge = EdgeAttributes(base_cost=10.0, status=EdgeStatus.FORBIDDEN)

print("Arete libre      :", free_edge.static_cost)
print("Arete surchargee :", surcharged_edge.static_cost)
print("Arete interdite  :", forbidden_edge.static_cost)

Arete libre      : 10.0
Arete surchargee : 12.5
Arete interdite  : inf


## 3. `GraphGenerationConfig`

Dataclass frozen avec beaucoup de champs optionnels. Les validations sont faites dans `__post_init__`.

Le principe : l'utilisateur donne peu, le reste est **derive** (`resolved_*` properties).

In [4]:
config = GraphGenerationConfig(node_count=12, seed=42)
print("edge_density fournie        :", config.edge_density)
print("auto_density_profile        :", config.auto_density_profile)
print("resolved_edge_density       :", config.resolved_edge_density)
print("resolved_min_base_density   :", config.resolved_min_base_density)
print("resolved_max_base_density   :", config.resolved_max_base_density)
print("resolved_min_residual_density :", config.resolved_min_residual_density)
print("resolved_max_residual_density :", config.resolved_max_residual_density)

edge_density fournie        : None
auto_density_profile        : True
resolved_edge_density       : 0.45
resolved_min_base_density   : 0.3
resolved_max_base_density   : 0.6
resolved_min_residual_density : 0.22
resolved_max_residual_density : 0.5


## 4. Profil de densite automatique

Quand `auto_density_profile=True` et qu'aucune borne n'est specifiee, `_recommended_density_profile(node_count)` applique :

| `node_count` | min base | max base | min residuelle | max residuelle |
|---|---|---|---|---|
| `<= 10`  | 0.45 | 0.75 | 0.35 | 0.65 |
| `<= 25`  | 0.30 | 0.60 | 0.22 | 0.50 |
| `> 25`   | 0.18 | 0.40 | 0.12 | 0.30 |

Pourquoi des profils par taille ? Pour un graphe a 5 sommets, 40 % de densite = 4 aretes. Pour un graphe a 100 sommets, 40 % = presque 2000 aretes. Ces echelles demandent des bornes differentes sinon on obtient des graphes soit trop creux (inexploitables), soit trop denses (trivialement resolus).

In [5]:
for n in [6, 15, 30, 60]:
    cfg = GraphGenerationConfig(node_count=n, seed=0)
    print(
        f"n={n:>3} | base ~ [{cfg.resolved_min_base_density}, {cfg.resolved_max_base_density}]"
        f"  | res ~ [{cfg.resolved_min_residual_density}, {cfg.resolved_max_residual_density}]"
        f"  | cible = {cfg.resolved_edge_density}"
    )

n=  6 | base ~ [0.45, 0.75]  | res ~ [0.35, 0.65]  | cible = 0.6
n= 15 | base ~ [0.3, 0.6]  | res ~ [0.22, 0.5]  | cible = 0.45
n= 30 | base ~ [0.18, 0.4]  | res ~ [0.12, 0.3]  | cible = 0.29
n= 60 | base ~ [0.18, 0.4]  | res ~ [0.12, 0.3]  | cible = 0.29


## 5. Seuils derives (formules internes)

En plus des densites, la config deduit trois seuils utilises par les validateurs.

**Degre moyen residuel minimal** :

```text
resolved_min_average_residual_degree = max(2, min_residual_density * (n - 1) * 0.85)
```

On impose au moins 2 voisins en moyenne. Le facteur 0.85 laisse une marge par rapport a la densite minimale pour eviter les rejets a la limite.

**Densite dynamique minimale** :

```text
resolved_dynamic_min_density = max(0.10, min_residual_density * 0.85)
```

Lors de la simulation, on tolere une densite un peu plus basse que le residuel statique (coupe temporaire), mais on garde un plancher de 10 %.

**Degre moyen actif minimal** :

```text
resolved_dynamic_min_average_degree = max(2, min_avg_residual_degree * 0.85)
```

In [6]:
cfg = GraphGenerationConfig(node_count=15, seed=0)
print("min_avg_residual_degree     :", cfg.resolved_min_average_residual_degree)
print("dynamic_min_density         :", cfg.resolved_dynamic_min_density)
print("dynamic_min_average_degree  :", cfg.resolved_dynamic_min_average_degree)

min_avg_residual_degree     : 2.62
dynamic_min_density         : 0.187
dynamic_min_average_degree  : 2.23


## 6. `GraphInstance`

Dataclass (mutable) qui regroupe l'instance complete : config + coordonnees + trois matrices + aretes + demandes + chemins reels.

Proprietes utiles :

- `base_density`, `residual_density`, `residual_average_degree` : metriques reelles ;
- `uniform_demand` : valeur de la demande uniforme ;
- `minimum_route_count = ceil(total_demand / vehicle_capacity)` : borne basse theorique sur le nombre de tournees ;
- `edge(u, v)` : raccourci pour lire l'`EdgeAttributes` normalisee (cle `(min(u,v), max(u,v))`) ;
- `summary()` : dict consolide pour debug et reporting.

In [7]:
from cesipath.graph_generator import GraphGenerator

instance = GraphGenerator(GraphGenerationConfig(node_count=8, seed=42)).generate()
print("Densites :", instance.base_density, instance.residual_density)
print("Degre moyen residuel :", instance.residual_average_degree)
print("Demande uniforme :", instance.uniform_demand)
print("Nombre minimal de tournees :", instance.minimum_route_count)

u, v = next(iter(instance.residual_edges))
print(f"Arete ({u}, {v}) :", instance.edge(u, v))

Densites : 0.5714285714285714 0.5
Degre moyen residuel : 3.5
Demande uniforme : 8
Nombre minimal de tournees : 2
Arete (0, 1) : EdgeAttributes(base_cost=41.48, status=<EdgeStatus.FREE: 'libre'>, static_surcharge=0.0)


## 7. `DynamicGraphSnapshot`

Represente l'etat du reseau a un tour de simulation `step`. Attributs :

- `edge_costs` : cout courant par arete (meme OFF, on garde le cout connu) ;
- `edge_availability` : dict `{(u,v): bool}`, True si l'arete est utilisable ce tour ;
- `residual_costs`, `completed_costs`, `completed_paths` : reconstruits apres chaque `advance()`.

Accesseurs pratiques : `edge_cost(u, v)`, `is_available(u, v)`, et la propriete `active_edge_count`.

In [8]:
from cesipath.dynamic_network import DynamicNetworkSimulator

sim = DynamicNetworkSimulator(instance, seed=42)
snap = sim.initialize_snapshot()
print("Type :", type(snap).__name__)
print("Step :", snap.step)
print("Aretes actives :", snap.active_edge_count)
print("Cout (0, 1) :", snap.edge_cost(0, 1))
print("Disponible (0, 1) ?", snap.is_available(0, 1))

Type : DynamicGraphSnapshot
Step : 0
Aretes actives : 14
Cout (0, 1) : 41.48
Disponible (0, 1) ? True


## 8. A retenir

Le fichier `models.py` ne fait **aucun** calcul metier. Il ne fait que :

- valider les entrees dans `__post_init__` ;
- exposer des proprietes derivees.

Toute la logique est dans `graph_generator.py`, `dynamic_network.py`, `metric_closure.py` et `validators.py`. Garder cette separation permet de brancher un jour un chargeur OpenStreetMap sans toucher aux modeles.